# AI-Lake — Multimodal Tables (Phase 8)

Demonstrates **N vector columns**, **modality tagging**, and **cross-modal RRF search**.

| Section | Feature |
|---|---|
| 1 | Setup + fixture (text + image embeddings, 200 rows) |
| 2 | `VectorColSpec` + modality tags (`text` / `image`) |
| 3 | `write_batch_multi` — write N columns in one call |
| 4 | Inspect Iceberg metadata (`ailake.modality-*`, `ailake.dim-*`) |
| 5 | Single-column search (text-only, image-only) |
| 6 | `search_multimodal` — RRF fusion across columns |
| 7 | Weight ablation — 100/0 → 70/30 → 50/50 → 0/100 |
| 8 | Pre-generated fixture from `init_demo.py` |
| 9 | `MultimodalContextSchema` column name constants |
| 10 | Assemble multimodal context for LLMs |


## 0. Setup

In [ ]:
import io, json, os, pathlib
import numpy as np
import pandas as pd
import pyarrow as pa
import ailake
from ailake import TableWriter, VectorColSpec

BASE         = pathlib.Path(os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')).parent
MM_NB_PATH   = str(BASE / 'nb07_multimodal')          # notebook-local table
MM_DEMO_PATH = os.environ.get('DEMO_MULTIMODAL_PATH', str(BASE / 'ailake_multimodal'))  # pre-gen

TEXT_DIM  = 32   # text embedding dimension
IMAGE_DIM = 16   # synthetic CLIP-like image embeddings
N         = 60   # number of demo rows

rng = np.random.default_rng(seed=2026)

SUBJECTS = ['cat', 'dog', 'car', 'tree', 'sky', 'ocean', 'mountain', 'city', 'forest', 'river']
TEMPLATES = [
    'A photo of a {s} in daylight.',
    'Close-up image of a {s} with blurred background.',
    'Aerial view of {s}s near the coast.',
    'Black-and-white photograph of a {s}.',
    'Digital artwork depicting a {s} at sunset.',
    'Documentary footage: a {s} in its natural habitat.',
]
texts = [TEMPLATES[i % len(TEMPLATES)].format(s=SUBJECTS[i % len(SUBJECTS)]) + f' (id={i})'
         for i in range(N)]

text_embs  = rng.standard_normal((N, TEXT_DIM)).astype(np.float32)
text_embs /= np.linalg.norm(text_embs, axis=1, keepdims=True)

image_embs  = rng.standard_normal((N, IMAGE_DIM)).astype(np.float32)
image_embs /= np.linalg.norm(image_embs, axis=1, keepdims=True)

print(f'texts      : {len(texts)}')
print(f'text_embs  : shape={text_embs.shape}  dtype={text_embs.dtype}')
print(f'image_embs : shape={image_embs.shape}  dtype={image_embs.dtype}')


## 1. `VectorColSpec` — declare vector columns

`VectorColSpec(column, dim, metric, modality)` describes one vector column.

| Arg | Description | Example |
|---|---|---|
| `column` | Parquet column name | `"embedding"` |
| `dim` | Embedding dimension | `1536` |
| `metric` | Distance metric | `"cosine"` |
| `modality` | Optional tag | `"text"` / `"image"` / `"audio"` / `"video"` |

Modality is stored as `ailake.modality-<column>` in Iceberg table properties,
enabling readers to select the right HNSW index without inspecting data.


In [ ]:
text_spec  = VectorColSpec('embedding',       TEXT_DIM,  'cosine', 'text')
image_spec = VectorColSpec('image_embedding', IMAGE_DIM, 'cosine', 'image')

print(f'text_spec  : column={text_spec.column!r}  dim={text_spec.dim}  metric={text_spec.metric!r}  modality={text_spec.modality!r}')
print(f'image_spec : column={image_spec.column!r}  dim={image_spec.dim}  metric={image_spec.metric!r}  modality={image_spec.modality!r}')


## 2. `write_batch_multi` — write N columns in one call

Each column gets its own HNSW index in the AILK section of the Parquet footer.
The AILK section is **invisible to standard Parquet/Iceberg readers** — they see
only two opaque `FIXED_LEN_BYTE_ARRAY` columns and read them as bytes.


In [ ]:
w = TableWriter(MM_NB_PATH, dim=TEXT_DIM, metric='cosine')
w.write_batch_multi(
    texts,
    [
        (text_spec,  text_embs.tolist()),
        (image_spec, image_embs.tolist()),
    ],
)
snap = w.commit()
print(f'Snapshot ID: {snap}')
print(f'Table path : {MM_NB_PATH}')


## 3. Inspect Iceberg metadata

After `write_batch_multi` + `commit`, the Iceberg `metadata.json` contains
`ailake.*` properties for both columns:

- `ailake.modality-embedding` = `"text"`
- `ailake.modality-image_embedding` = `"image"`
- `ailake.dim-image_embedding` = `"16"`
- `ailake.metric-image_embedding` = `"cosine"`

The primary column (`embedding`) uses the legacy keys `ailake.vector-dim` and
`ailake.vector-metric`; secondary columns have per-column keys.


In [ ]:
meta_dir = pathlib.Path(MM_NB_PATH) / 'default' / 'table' / 'metadata'
hint     = (meta_dir / 'version-hint.text').read_text().strip()
meta     = json.loads((meta_dir / f'v{hint}.metadata.json').read_text())
props    = meta.get('properties', {})

print('AI-Lake table properties (ailake.* keys):')
for k in sorted(props):
    if k.startswith('ailake.'):
        print(f'  {k:45s} = {props[k]}')


## 4. Single-column search

`ailake.search()` targets one column. With a multimodal table you can choose:
- `vector_column='embedding'` — search by text similarity
- `vector_column='image_embedding'` — search by image similarity

Both return the same `row_id / distance / file` schema.


In [ ]:
text_query  = text_embs[0].tolist()
image_query = image_embs[0].tolist()

# Text-only search (default column 'embedding')
r_text = ailake.search(MM_NB_PATH, text_query, top_k=5).to_pandas()
print('Text-only (column=embedding):')
print(r_text[['row_id', 'distance']].to_string(index=False))


In [ ]:
# Image-only search — search by image embedding column
# Uses ailake.search with vector_column override via search_multimodal (weight=1)
r_image = ailake.search_multimodal(
    MM_NB_PATH,
    queries=[('image_embedding', image_query, 1.0)],
    top_k=5,
)
df_img = pd.DataFrame(r_image)
print('Image-only (column=image_embedding):')
print(df_img[['row_id', 'rrf_score']].to_string(index=False))


## 5. `search_multimodal` — RRF fusion

RRF score per row: `Σ weight_i / (60 + rank_i)` — higher is better.

- Each column searched independently by its HNSW (respects per-column dim automatically)
- Ranks fused via Reciprocal Rank Fusion
- Results ordered by descending `rrf_score`


In [ ]:
# Equal weight RRF (0.5 / 0.5)
results_50_50 = ailake.search_multimodal(
    MM_NB_PATH,
    queries=[
        ('embedding',       text_query,  0.5),
        ('image_embedding', image_query, 0.5),
    ],
    top_k=5,
)
df_50_50 = pd.DataFrame(results_50_50)
print('RRF 50%/50%:')
df_50_50[['row_id', 'rrf_score']]


In [ ]:
# Text-dominant (0.7 / 0.3) — typical RAG with image context
results_70_30 = ailake.search_multimodal(
    MM_NB_PATH,
    queries=[
        ('embedding',       text_query,  0.7),
        ('image_embedding', image_query, 0.3),
    ],
    top_k=5,
)
df_70_30 = pd.DataFrame(results_70_30)
print('RRF 70%/30%:')
df_70_30[['row_id', 'rrf_score']]


## 6. Weight ablation

Compare top-5 rankings across five weight configurations to see how fusion
order changes as the balance shifts from text-dominant to image-dominant.


In [ ]:
configs = [
    (1.0, 0.0, 'text only'),
    (0.7, 0.3, 'text-dominant'),
    (0.5, 0.5, 'equal weight'),
    (0.3, 0.7, 'image-dominant'),
    (0.0, 1.0, 'image only'),
]

rows = []
for tw, iw, label in configs:
    r = ailake.search_multimodal(
        MM_NB_PATH,
        queries=[('embedding', text_query, tw), ('image_embedding', image_query, iw)],
        top_k=5,
    )
    rows.append({'config': label, 'tw': tw, 'iw': iw, 'top5_ids': [x['row_id'] for x in r]})

df_ablation = pd.DataFrame(rows)
print('Weight ablation — top-5 row_ids:')
df_ablation


## 7. Pre-generated fixture

`init_demo.py` wrote `ailake_multimodal` at container startup (200 rows,
`embedding` dim=32 + `image_embedding` dim=16).  Access it via `DEMO_MULTIMODAL_PATH`.


In [ ]:
query_path = pathlib.Path(os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')).parent / 'demo_query.json'
with open(query_path) as f:
    demo = json.load(f)

mm_info = demo.get('multimodal', {})
print(f'Pre-gen multimodal table : {demo["table_paths"]["multimodal"]}')
print(f'  text column  : {mm_info.get("text_column")}  dim={mm_info.get("text_dim")}')
print(f'  image column : {mm_info.get("image_column")}  dim={mm_info.get("image_dim")}')


In [ ]:
# Search the pre-gen fixture using a random query
tq_pregen  = np.random.default_rng(99).standard_normal(mm_info['text_dim']).astype(np.float32)
iq_pregen  = np.random.default_rng(100).standard_normal(mm_info['image_dim']).astype(np.float32)
tq_pregen /= np.linalg.norm(tq_pregen)
iq_pregen /= np.linalg.norm(iq_pregen)

r_pregen = ailake.search_multimodal(
    MM_DEMO_PATH,
    queries=[
        (mm_info['text_column'],  tq_pregen.tolist(), 0.6),
        (mm_info['image_column'], iq_pregen.tolist(), 0.4),
    ],
    top_k=5,
)
print('Pre-gen fixture RRF (60%/40%):')
pd.DataFrame(r_pregen)[['row_id', 'rrf_score']]


## 8. `MultimodalContextSchema` — column name constants

Use canonical column names from `ailake.MULTIMODAL_COLUMNS` to avoid
hard-coded strings. These names align with what `ContextAssembler` expects.

> **Design principle**: AI-Lake stores **URIs** and **embeddings**, never raw media bytes.
> Media files live in object storage (`s3://`, `gs://`, `az://`, `https://`);
> the AI-Lake row points to them via `media_uri`.


In [ ]:
# Canonical column name constants (mirrored from ailake-core/src/schema.rs)
MULTIMODAL_COLUMNS = {
    'MEDIA_URI':        'media_uri',        # URI in object storage
    'MEDIA_MIME':       'media_mime',        # image/jpeg, audio/mpeg, …
    'MEDIA_CAPTION':    'media_caption',     # BLIP-2 / Whisper caption
    'IMAGE_EMBEDDING':  'image_embedding',   # CLIP/SigLIP dim=512
    'AUDIO_TRANSCRIPT': 'audio_transcript',  # Whisper transcript
    'THUMBNAIL_B64':    'thumbnail_b64',     # base64 JPEG ≤ 64×64
}

for name, val in MULTIMODAL_COLUMNS.items():
    print(f'  {name:22s} = {val!r}')


In [ ]:
# Example: build a full MultimodalContextSchema Arrow schema
from ailake.llm_columns import (
    CHUNK_ID, CHUNK_TEXT, EMBEDDING, CONTEXT_EMBEDDING,
    DOCUMENT_TITLE, SOURCE_URI,
) if hasattr(ailake, 'llm_columns') else (None,) * 6

schema = pa.schema([
    pa.field('chunk_id',          pa.string()),
    pa.field('chunk_text',        pa.string()),
    pa.field('document_title',    pa.string()),
    pa.field('source_uri',        pa.string()),
    # Vector columns
    pa.field('embedding',         pa.binary()),   # text F16 dim=1536
    pa.field('image_embedding',   pa.binary()),   # image F16 dim=512
    # Multimodal columns
    pa.field('media_uri',         pa.string()),
    pa.field('media_mime',        pa.string()),
    pa.field('media_caption',     pa.string()),
    pa.field('audio_transcript',  pa.string()),
    pa.field('thumbnail_b64',     pa.string()),
])
print('MultimodalContextSchema (Arrow):')
for f in schema:
    print(f'  {f.name:22s}: {f.type}')


## 9. Assemble multimodal context for LLMs

`ailake.assemble_context()` accepts any chunk dict including multimodal fields.
When `media_uri` is present, the assembler includes it in the XML block so
multimodal LLMs (Claude, GPT-4o) can optionally fetch the referenced media.


In [ ]:
chunks_mm = [
    {
        'document_id':    'img-001',
        'chunk_index':    0,
        'chunk_text':     'A photograph of a golden retriever playing fetch on a beach.',
        'document_title': 'Dog Photos Dataset',
        'source_uri':     's3://my-lake/photos/img-001.jpg',
        'distance':       0.08,
        'media_uri':      's3://my-lake/photos/img-001.jpg',
        'media_mime':     'image/jpeg',
        'media_caption':  'Golden retriever fetching a ball at the beach.',
    },
    {
        'document_id':    'img-002',
        'chunk_index':    0,
        'chunk_text':     'Aerial view of a mountain range covered in snow.',
        'document_title': 'Landscape Photography',
        'source_uri':     's3://my-lake/photos/img-002.jpg',
        'distance':       0.15,
        'media_uri':      's3://my-lake/photos/img-002.jpg',
        'media_mime':     'image/jpeg',
        'media_caption':  'Snow-covered mountain peaks from above.',
    },
]

ctx_xml = ailake.assemble_context(chunks=chunks_mm, max_tokens=2048, dedup_threshold=0.05)
print(ctx_xml)


## Next Steps

| Resource | Location |
|---|---|
| Full SDK walkthrough | `01_ailake_demo.ipynb` (sections 1-23) |
| DuckDB compat | `02_duckdb.ipynb` |
| Spark compat | `03_spark.ipynb` |
| Trino plugin (engines profile) | `04_trino.ipynb` |
| Phase 8 CLAUDE.md | `CLAUDE.md §8` — implementation notes |
| Rust API | `ailake-py/src/lib.rs` `write_batch_multi`, `search_multimodal` |
| Column constants (Rust) | `ailake-core/src/schema.rs` `multimodal_columns` module |
